# KG1 v74 CURRICULUM - Evolucao sobre v73 DEFINITIVE (target 0.86+)

## Pre-requisitos:
- v73 DEFINITIVE atingiu >=0.85 Kaggle
- Dataset `felipesp1983/kg1-v74-training` gerado via `scripts/prepare_v74_dataset.py`

## Melhorias vs v73:
1. Dataset 10K multi-source (kienngx + huikang + ritwika + konbu17)
2. Curriculum learning (easy -> hard)
3. Two-stage SFT: curriculum -> uniform
4. Min-logprob callback (huikang Open Progress Prize pattern)
5. 3 epochs + cosine restarts

## Score target: 0.86-0.87 (+0.005-0.015 sobre v73 baseline)

In [ ]:
# Cell 1: Install (reusa estrategia v73 DEFINITIVE)
%%capture
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl>=0.16' 'peft>=0.18.1' accelerate bitsandbytes
!pip install -q 'transformers>=4.55' datasets hf_transfer


In [ ]:
# Cell 2: Disable broken unsloth patch + env + GPU check (mesmo v73)
import os, subprocess, sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["UNSLOTH_IS_PRESENT"] = "0"

try:
    import unsloth_zoo
    uzoo_path = os.path.dirname(unsloth_zoo.__file__)
    misc_path = f"{uzoo_path}/temporary_patches/misc.py"
    if os.path.exists(misc_path):
        with open(misc_path, "r") as f:
            content = f.read()
        patch_line = "TEMPORARY_PATCHES.append(patch_merge_quantization_configs)"
        if patch_line in content and f"# DISABLED: {patch_line}" not in content:
            new_content = content.replace(patch_line, f"# DISABLED: {patch_line}")
            with open(misc_path, "w") as f:
                f.write(new_content)
            print(f"[OK] Disabled broken patch")
            for mod_name in list(sys.modules):
                if "unsloth" in mod_name.lower(): del sys.modules[mod_name]
except ImportError: pass

import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} VRAM: {vram:.1f}GB")
assert vram >= 70, "Precisa H100 ou A100 80GB"

from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))


In [ ]:
# Cell 3: Drive + HF + Kaggle secrets (mesmo v73)
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_KEY")
except Exception:
    HF_TOKEN = userdata.get("HF_TOKEN", "")
assert HF_TOKEN.startswith("hf_")
os.environ["HF_TOKEN"] = HF_TOKEN

CKPT_DIR = "/content/drive/MyDrive/kg1_v74_curriculum"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"[OK] CKPT_DIR: {CKPT_DIR}")


In [ ]:
# Cell 4: Load Nemotron-30B via Unsloth (mesmo v73 DEFINITIVE)
import sys, torch, gc
gc.collect(); torch.cuda.empty_cache()

for mod_name in list(sys.modules):
    if "unsloth" in mod_name.lower(): del sys.modules[mod_name]

import unsloth
from unsloth import FastLanguageModel

MAX_SEQ = 2048
MODEL_ID = "unsloth/Nemotron-3-Nano-30B-A3B"

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
)

if tok.pad_token is None: tok.pad_token = tok.eos_token
print(f"[OK] GPU mem: {torch.cuda.memory_allocated()/1e9:.1f}GB")


In [ ]:
# Cell 5: LoRA kienngx sem MoE (mesmo v73)
from unsloth import FastLanguageModel

TARGET_MODULES = ["in_proj", "out_proj", "q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=32, lora_dropout=0.05,
    bias="none",
    target_modules=TARGET_MODULES,
    target_parameters=[],
    use_rslora=False, use_dora=False,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()


In [ ]:
# Cell 6 v74: Multi-source curriculum dataset (10K curated)
import os, pandas as pd
from datasets import load_dataset, Dataset

# Tentar carregar dataset v74 pre-preparado (via scripts/prepare_v74_dataset.py)
try:
    ds_v74 = load_dataset("felipesp1983/kg1-v74-training", split="train", token=HF_TOKEN)
    df = ds_v74.to_pandas()
    print(f"[OK] V74 dataset loaded: {len(df)} rows")
except Exception as e:
    print(f"WARN: V74 dataset nao disponivel ({e})")
    print("Fallback: kienngx baseline (1200 train.csv seed=42)")
    # Fallback: usar baseline kienngx
    import shutil
    if not os.path.exists("/content/drive/MyDrive/kg1_train.csv"):
        os.system("kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/")
        os.system("unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true")
        shutil.copy("/content/train.csv", "/content/drive/MyDrive/kg1_train.csv")
    df = pd.read_csv("/content/drive/MyDrive/kg1_train.csv")
    df = df.sample(n=1200, random_state=42).reset_index(drop=True)
    df["difficulty"] = 0.5  # unknown
    df["source"] = "kienngx_baseline"

# CURRICULUM: sort easy -> hard via difficulty
if "difficulty" in df.columns:
    df = df.sort_values("difficulty").reset_index(drop=True)
    print(f"Curriculum: difficulty range {df['difficulty'].min():.2f} - {df['difficulty'].max():.2f}")

PROMPT_COL = "prompt" if "prompt" in df.columns else "problem"
ANSWER_COL = "answer" if "answer" in df.columns else "solution"
PROMPT_SUFFIX = chr(10) + "Put your final answer inside \\boxed{}."

def format_v74(row):
    user_msg = str(row[PROMPT_COL]) + PROMPT_SUFFIX
    assistant_msg = str(row[ANSWER_COL])
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

keep_cols = [c for c in df.columns if c not in ["text"]]
ds_train = Dataset.from_pandas(df[keep_cols]).map(format_v74, num_proc=2, remove_columns=keep_cols)
print(f"[OK] Formatted: {len(ds_train)} examples (curriculum ordered)")


In [ ]:
# Cell 7 v74: Two-stage SFT (curriculum -> uniform) + min-logprob callback
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import threading, time, trl, torch, os

# Callback: min-logprob monitoring (huikang Open Progress Prize)
class MinLogprobCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            # Min-logprob proxy: loss is neg log likelihood, track percentiles
            if state.global_step % 50 == 0:
                print(f"[STEP {state.global_step}] loss={logs.get('loss', 0):.4f}")

# STAGE 1: CURRICULUM (shuffle=False, epochs=2)
STAGE1_DIR = f"{CKPT_DIR}/stage1_curriculum"
os.makedirs(STAGE1_DIR, exist_ok=True)

args_stage1 = SFTConfig(
    output_dir=STAGE1_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    optim="adamw_torch",
    max_length=MAX_SEQ,
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=False,
    max_grad_norm=1.0,
    report_to="none",
    seed=42,
)

trainer1 = SFTTrainer(
    model=model, train_dataset=ds_train,
    args=args_stage1, processing_class=tok,
    callbacks=[MinLogprobCallback()],
)

# Force sequential sampler para curriculum respect order
class SequentialSampler:
    def __init__(self, data): self.data = data
    def __iter__(self): return iter(range(len(self.data)))
    def __len__(self): return len(self.data)
trainer1._get_train_sampler = lambda: SequentialSampler(trainer1.train_dataset)

def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated()/1e9
            p = torch.cuda.max_memory_allocated()/1e9
            print(f"[MEM] current={m:.1f}GB peak={p:.1f}GB")
        except: pass
        time.sleep(300)
threading.Thread(target=monitor_mem, daemon=True).start()

print("Stage 1: CURRICULUM easy->hard (2 epochs)...")
stats1 = trainer1.train()
print(f"Stage 1 done. Loss: {stats1.training_loss:.4f}")

# STAGE 2: UNIFORM SHUFFLE (1 epoch consolidation)
STAGE2_DIR = f"{CKPT_DIR}/stage2_uniform"
os.makedirs(STAGE2_DIR, exist_ok=True)

args_stage2 = SFTConfig(
    output_dir=STAGE2_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,  # menor LR para consolidacao
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    bf16=True,
    optim="adamw_torch",
    max_length=MAX_SEQ,
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=False,
    max_grad_norm=1.0,
    report_to="none",
    seed=43,  # seed diferente = shuffle
)

trainer2 = SFTTrainer(
    model=model, train_dataset=ds_train,
    args=args_stage2, processing_class=tok,
)

print("\nStage 2: UNIFORM shuffle (1 epoch consolidation)...")
stats2 = trainer2.train()
print(f"Stage 2 done. Loss: {stats2.training_loss:.4f}")
print(f"\nFinal loss v74: {stats2.training_loss:.4f}")
trainer = trainer2  # para Cell 8
stats = stats2


In [ ]:
# Cell 8: Save + validate + submission.zip + HF upload
import os, json, zipfile
from huggingface_hub import HfApi

FINAL_DIR = f"{CKPT_DIR}/final_adapter_v74"
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)

with open(f"{FINAL_DIR}/adapter_config.json") as f:
    cfg = json.load(f)
tm = cfg.get("target_modules", [])
rank = cfg.get("r", cfg.get("lora_rank", 0))

errors = []
if not isinstance(tm, list): errors.append("target_modules invalid")
else:
    if "in_proj" not in tm: errors.append("missing in_proj")
    if "gate_proj" in tm: errors.append("has gate_proj")
    if "x_proj" in tm: errors.append("has x_proj")
if rank > 32: errors.append(f"rank {rank} > 32")

if errors:
    print(f"!!! GATE FAIL: {errors}")
else:
    print("[OK] Gate pass")

SUBMISSION_ZIP = f"{CKPT_DIR}/submission_v74.zip"
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["adapter_config.json", "adapter_model.safetensors"]:
        src = os.path.join(FINAL_DIR, f)
        if os.path.exists(src):
            zf.write(src, arcname=f)
            print(f"  added: {f} ({os.path.getsize(src)/1e6:.1f} MB)")

api = HfApi(token=HF_TOKEN)
REPO = "felipesp1983/kg1-nemotron-lora-v74-curriculum"
try:
    api.create_repo(REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO, path_in_repo="final")
    api.upload_file(path_or_fileobj=SUBMISSION_ZIP, repo_id=REPO, path_in_repo="submission.zip")
    print(f"[OK] HF: {REPO}")
except Exception as e:
    print(f"HF upload: {e}")

print(f"\nScore target v74: 0.86-0.87 (+0.005-0.015 sobre v73)")
print(f"Pre-submit: python scripts/local_score.py --adapter {REPO} --target-score 0.84")
print(f"Submit: python scripts/submit_kaggle.py --hf-repo {REPO} --message 'v74 curriculum two-stage loss {stats.training_loss:.3f}'")
